# 10.6 視覺與文字維度不同怎麼接？

# 第 10 章：圖片如何進入文字模型

前置：embedding、序列與Linear。圖片入口先用16×16合成RGB，不需外部權重；圖片是否被理解必須等語言／視覺基礎能力訓練後測試。程式在tiny_perceptron/multimodal.py。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一個圖片佔位格，展開成三個向量**

展開 labels 同步加 ignored，再做下一字 shift。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/modal_expand.svg")))

**先想一想：**接頭維度正確，等於模型知道每個feature的意思嗎？

projector是兩種feature介面的轉接頭。輸入Dv維，輸出文字模型D維；尺寸相容只是必要條件，不表示語意已對齊。

**把直覺寫成數學：**projector W:[D,Dv]，輸出[B,N,D]。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
vision = torch.randn(1, 16, 8)
projector = nn.Linear(8, 12)
language_features = projector(vision)
print(language_features.shape)
assert language_features.shape == (1, 16, 12)

**如何讀結果：**projector需要合適訓練訊號；11.1比較隨機接頭與真的內容能力。

**只改一件事：**只改D，核對文字embedding也需匹配。
